In [ ]:
!pip install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.0 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.11.0
    Uninstalling google-genai-2.11.0:
      Successfully uninstalled google-genai-2.11.0


In [ ]:
import os
import time
import glob
from google.colab import auth
from googleapiclient.discovery import build
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from google.colab import userdata

# 1. Authenticate with Google Drive & Docs
auth.authenticate_user()
docs_service = build('docs', 'v1')
drive_service = build('drive', 'v3')

# 2. Setup New Gemini Client
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# TODO: Ensure this is your actual Google Doc Template ID
TEMPLATE_DOCUMENT_ID = "1gbZPhUeQ5cw-T3tL51mvwZ0RoOSEz8F1Uoa27gBxPw0"

# --- NEW: Find all MP4 video files in the current Colab directory ---
video_filenames = glob.glob("*.mp4")

if not video_filenames:
    raise FileNotFoundError("No .mp4 files found. Please drag and drop your TikTok videos into the Colab file explorer.")

# 3. Pydantic Schema optimized for bulk video transcription
class MeetingSummary(BaseModel):
    document_title: str = Field(
        description="A short, descriptive title for the document based on the business info extracted from the videos. Format: TikTok-Context-Dump-YYYY-MM-DD"
    )
    add_to_context_window: str = Field(
        description="The complete, concatenated transcripts of all provided videos. "
                    "Must include every piece of spoken information, text on screen, and business detail mentioned. "
                    "Do not summarize. Provide the raw, exhaustive transcript data."
    )

# 4. Prompt optimized for raw transcription extraction
prompt = """
Task: Analyze this batch of TikTok videos and extract the absolute full, raw transcript and any on-screen text containing business information.

Focus Framework:
- Transcribe everything spoken in every video.
- Capture any text displayed on the screen (prices, URLs, addresses, product names).
- Clearly separate the transcript of each video so it is readable.

Constraint: DO NOT summarize or truncate. I need the raw, uncompressed text data to feed into a chatbot's context window. Extend the text as long as necessary to capture everything.
"""

# 5. Upload all videos via the New SDK
print(f"Found {len(video_filenames)} videos. Starting upload process...")
uploaded_files = []

for filename in video_filenames:
    print(f"Uploading {filename}...")
    video_file = client.files.upload(file=filename)

    # Wait for processing
    while video_file.state.name == 'PROCESSING':
        print('.', end='')
        time.sleep(5)
        video_file = client.files.get(name=video_file.name)

    if video_file.state.name == 'FAILED':
        print(f"\nWarning: Processing failed for {filename}. Skipping.")
        continue

    print(f"\n{filename} ready!")
    uploaded_files.append(video_file)

if not uploaded_files:
    raise ValueError("No videos were successfully processed by the Gemini API.")

# 6. Generate Content using Structured Output
print("Generating exhaustive combined transcript from all videos...")

# Pass the text prompt AND the list of all uploaded video files together
contents_payload = [prompt] + uploaded_files

response = client.models.generate_content(
    model='gemini-flash-latest',
    contents=contents_payload,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MeetingSummary,
        temperature=0.1
    )
)

summary_data = response.parsed
print(f"Success! Transcripts parsed. Title generated: {summary_data.document_title}")

# 7. Generate the Google Doc
print(f"Creating new Google Doc from template: {summary_data.document_title}...")
copied_file = drive_service.files().copy(
    fileId=TEMPLATE_DOCUMENT_ID,
    body={'name': summary_data.document_title}
).execute()
new_doc_id = copied_file.get('id')

# Build the batchUpdate requests array targeting the single placeholder string
requests = [
    {
        'replaceAllText': {
            'containsText': {
                'text': '{{ADD_TO_CONTEXT_WINDOW}}',
                'matchCase': True
            },
            'replaceText': summary_data.add_to_context_window
        }
    }
]

# Execute the batch update on the Google Docs API
docs_service.documents().batchUpdate(
    documentId=new_doc_id,
    body={'requests': requests}
).execute()

print(f"\nDocument generated successfully! Access your raw chatbot context text here:")
print(f"https://docs.google.com/document/d/{new_doc_id}/edit")

Found 13 videos. Starting upload process...
Uploading 423dfcef97e580c5acc33d38e52564e1.mp4...
.
423dfcef97e580c5acc33d38e52564e1.mp4 ready!
Uploading 321c7b6b5d53fe3a68c3961f38ff5470.mp4...
.
321c7b6b5d53fe3a68c3961f38ff5470.mp4 ready!
Uploading 4a02f6e89be654d8e3623723d22dec9e.mp4...
.
4a02f6e89be654d8e3623723d22dec9e.mp4 ready!
Uploading a43235f79bf788d0bb0497d2074c6ff4.mp4...
.
a43235f79bf788d0bb0497d2074c6ff4.mp4 ready!
Uploading 247ec4c16ee246c5a978e862a8e69a0a.mp4...
.
247ec4c16ee246c5a978e862a8e69a0a.mp4 ready!
Uploading 49b37eeeaca2416ade203c2fc2059075.mp4...
..
49b37eeeaca2416ade203c2fc2059075.mp4 ready!
Uploading 48d7361d3152f50c7eff9c1fc52eabc4.mp4...
.
48d7361d3152f50c7eff9c1fc52eabc4.mp4 ready!
Uploading 7ece9b691d3adbf3fdd9a23df0bf6917.mp4...
.
7ece9b691d3adbf3fdd9a23df0bf6917.mp4 ready!
Uploading 9bc164197a429f3254ed29f23bfccb26.mp4...
.
9bc164197a429f3254ed29f23bfccb26.mp4 ready!
Uploading 9f830b24bea489c34601d9534c139727.mp4...
.
9f830b24bea489c34601d9534c139727.mp4 re

Success! Transcripts parsed. Title generated: TikTok-Context-Dump-2024-04-25
Creating new Google Doc from template: TikTok-Context-Dump-2024-04-25...



Document generated successfully! Access your raw chatbot context text here:
https://docs.google.com/document/d/1WD-1aO9RM7Fvn48_0mnrbBcI7h1IA-WhFRQhRXxanug/edit
